# Smart Elevator CV — YOLOv8 v2 Eğitimi + Sınıf-Tabanlı Alan Hesabı

**Tarih:** 2026-05-07  
**Bu notebook'un v1'den farkı:**
1. `best.pt` (v1) Drive üzerinde **`best_v1_backup_2026-05-07.pt`** olarak yedeklenir.
2. Yeni birleşik dataset (LASTDATASET dahil, 13 386 train img) üzerinde **`elevator_v2`** adıyla eğitim yapılır.
3. Eğitim sonrası **sınıf-tabanlı alan hesabı (class-count × ortalama footprint)** modülü çalıştırılır.

## Neden "sınıf-sayısı × ortalama alan"?
Asansör CCTV'si köşeye monte fisheye lens olduğu için **kameraya yakın kişilerin sadece kafa+omuz** kısmı görüntüye giriyor — full-body bbox fiziksel olarak imkansız. Bu yüzden:
- bbox piksel alanı yerine **tespit edilen sınıfların standart antropometrik footprint'lerini** kullanıyoruz.
- Bu yaklaşım **TS EN 81-20 / ISO 8100** asansör standartlarıyla uyumlu (kişi başı 0.20 m² minimum yer).
- İleride homografi kalibrasyonu yapılırsa `src/perception/occupancy.py` içindeki `FootprintOccupancy` veya `BEVMaskOccupancy`'ye geçilebilir (kod hazır, yalnızca homography matrix gerekiyor).

## Önkoşul (yerelde 1 kez)
1. `python -m scripts.package_for_colab` → `Desktop/colab_upload/` altında `code.zip` ve `dataset.zip` üretir.
2. Her iki ZIP'i **`MyDrive/Capstone/`** klasörüne yükle.
3. Bu notebook'u Drive'da bul → sağ tık → Open with → Google Colaboratory.

## 1. Drive'ı bağla ve ZIP'leri çıkar

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, zipfile, shutil
from datetime import date

BASE_DRIVE = '/content/drive/MyDrive/Capstone'
WORK = '/content/work'
REPO = f'{WORK}/Capstone_deneme_ai'
DATA = f'{WORK}/data/unified'
TODAY = date.today().isoformat()

assert os.path.exists(f'{BASE_DRIVE}/code.zip'),    f'code.zip bulunamadi: {BASE_DRIVE}/code.zip'
assert os.path.exists(f'{BASE_DRIVE}/dataset.zip'), f'dataset.zip bulunamadi: {BASE_DRIVE}/dataset.zip'

os.makedirs(REPO, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

with zipfile.ZipFile(f'{BASE_DRIVE}/code.zip') as z:
    z.extractall(REPO)
with zipfile.ZipFile(f'{BASE_DRIVE}/dataset.zip') as z:
    z.extractall(DATA)

# data.yaml'i Colab path'ine gore yeniden yaz (Windows yolu -> Linux yolu).
# Yerelde uretilen data.yaml mutlak Windows yolu icerir, Colab bunu bulamaz.
import yaml
yaml_path = f'{DATA}/data.yaml'
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)
old_path = data_cfg.get('path')
data_cfg['path'] = DATA
with open(yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(f'data.yaml path: {old_path!r} -> {DATA!r}')

%cd {REPO}
!ls

## 2. **ÖNCEKİ MODELİ YEDEKLE** (v1 best.pt → backup)

v1 modelinin Drive'daki konumu:  
`MyDrive/Capstone/models/runs/elevator_v1/weights/best.pt`

Yedek dosya adı: `best_v1_backup_<TARIH>.pt` (üzerine yazma riski sıfır).
Hem `models/weights/` altındaki kullanım kopyası hem de `runs/elevator_v1/weights/best.pt` kaynağı yedeklenir.

In [ ]:
# --- v1 best.pt yedekleme ---
from pathlib import Path

v1_candidates = [
    f'{BASE_DRIVE}/models/runs/elevator_v1/weights/best.pt',
    f'{BASE_DRIVE}/models/weights/best.pt',
]

v1_found = [p for p in v1_candidates if os.path.exists(p)]
print('Bulunan v1 dosyalari:')
for p in v1_found:
    sz_mb = os.path.getsize(p) / 1e6
    print(f'  • {p}  ({sz_mb:.1f} MB)')

if not v1_found:
    print('\n⚠ v1 best.pt bulunamadi — yedeklenecek model yok. Yeni egitime gecebilirsin.')
else:
    backup_dir = f'{BASE_DRIVE}/models/backups'
    os.makedirs(backup_dir, exist_ok=True)
    for src in v1_found:
        # Aynı dosyayı 2 kez yedeklememek için kaynak klasör adını adlandırmaya katıyoruz
        src_tag = Path(src).parent.parent.name  # elevator_v1 veya weights
        dst = f'{backup_dir}/best_v1_{src_tag}_backup_{TODAY}.pt'
        if os.path.exists(dst):
            print(f'  ⏭  Zaten yedekli: {dst}')
        else:
            shutil.copy2(src, dst)
            print(f'  ✅ Yedeklendi: {dst}')

print('\nYedek dizini icerigi:')
!ls -lh {BASE_DRIVE}/models/backups/ 2>/dev/null || echo '(dizin yok)'

## 3. Bağımlılıkları kur

In [ ]:
!pip install -q -r requirements.txt

## 4. GPU + Dataset doğrulaması

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!nvidia-smi 2>/dev/null | head -15

from src.dataset.audit import audit_yolo_dataset, print_audit
from src.dataset.unify import TARGET_CLASSES
print_audit(audit_yolo_dataset(DATA), class_names=TARGET_CLASSES)

## 5. Yeni modeli eğit (`elevator_v2`)

**Augmentation:** Yerelde idempotent şekilde uygulanmış (8 572 augment'li train img dahil 13 386 toplam) — burada YOLO'nun kendi runtime augmentation'ı default olarak yine devrede.

**Hyperparametreler v1 ile aynı tutuldu** — hangi değişikliğin etkili olduğunu görebilmek için tek değişken **dataset büyümesi** olarak bırakıldı. (LASTDATASET entegrasyonu + 957 boş etiket temizliği + leakage-free split)

In [ ]:
from ultralytics import YOLO

VARIANT = 'yolov8s.pt'
EPOCHS = 100
BATCH = 32
IMGSZ = 640

RUNS_DIR = f'{BASE_DRIVE}/models/runs'
os.makedirs(RUNS_DIR, exist_ok=True)

model = YOLO(VARIANT)
results = model.train(
    data=f'{DATA}/data.yaml',
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    lr0=0.01,
    patience=25,
    seed=42,
    project=RUNS_DIR,
    name='elevator_v2',
    plots=True,
    save_period=10,
    device=0 if torch.cuda.is_available() else 'cpu',
)
BEST_V2 = f'{results.save_dir}/weights/best.pt'
print('best.pt (v2):', BEST_V2)

## 6. Test set'te değerlendir

In [ ]:
from ultralytics import YOLO
model_v2 = YOLO(BEST_V2)
metrics = model_v2.val(data=f'{DATA}/data.yaml', split='test')
print('mAP50:    ', metrics.box.map50)
print('mAP50-95: ', metrics.box.map)
print('Per-class mAP50:', dict(zip(metrics.names.values(), metrics.box.maps.tolist())))

## 7. v2 best.pt'yi Drive ve repo'ya kopyala
Yedek (`best_v1_backup_*.pt`) korunur, yeni model **`best_v2.pt`** adıyla kaydedilir.  
Yerel/repo kullanımı için `models/weights/best.pt` üzerine yazılır (en güncel = v2).

In [ ]:
drive_v2 = f'{BASE_DRIVE}/models/weights/best_v2.pt'
drive_active = f'{BASE_DRIVE}/models/weights/best.pt'  # bunu yerel demo'lar kullanir
repo_dst = f'{REPO}/models/weights/best.pt'

for d in (drive_v2, drive_active, repo_dst):
    os.makedirs(os.path.dirname(d), exist_ok=True)
    shutil.copy2(BEST_V2, d)
    print('Copied:', d, f'({os.path.getsize(d) / 1e6:.1f} MB)')

## 8. **Sınıf-tabanlı alan hesabı** (literatüre dayalı)

### 8.1 Yöntem
Her tespit (`Detection`) için sınıfa özgü **standart footprint alanı** atanır ve toplam doluluk:

$$ A_{\text{occupied}} = \sum_{i} \text{count}(c_i) \cdot \bar{a}_{c_i} \quad,\quad \text{occupancy\%} = \min\!\left(\frac{A_{\text{occupied}}}{A_{\text{cabin}}}, 1.0\right) $$

### 8.2 Sınıf başına ortalama alan değerleri (literatür)

| Sınıf | Ortalama footprint (m²) | Kaynak / Gerekçe |
|---|---|---|
| **person** | **0.20** | TS EN 81-20:2020 §5.4.2.1.1 — "available car area per person" minimum 0.17–0.20 m². ISO 8100-1 aynı tabloyu kullanır. Asansör kapasite hesaplarındaki altın standart. |
| **stroller** | **0.45** | Tipik bebek arabası footprint'i ~90 × 50 cm = 0.45 m² (UPPAbaby Vista, Bugaboo Donkey vb. tek kademeli modeller). Çift kademeli (twin) modeller 0.60 m²'ye kadar çıkar — ortalama olarak 0.45 alındı. |
| **luggage** | **0.18** | Büyük valiz (75 × 50 cm) ≈ 0.375 m²; orta-küçük (55 × 35 cm) ≈ 0.19 m². Karışık dağılım için 0.18 m². IATA kabin bagajı standardı 56 × 36 × 23 cm baz alındı. |
| **box** | **0.20** | Orta boy karton (50 × 40 cm). e-ticaret/lojistik ortalama paket boyutu (Amazon "medium" box ≈ 45 × 35 cm). |

> ⚠ **Sınırlama notu:** Bu yaklaşım kişi/objelerin gerçek konumunu değil, **standart sınıf footprint'ini** kullanır. Asansör fiziksel olarak dolup yer kalmadığı durumlarda ($A_{\text{occupied}} > A_{\text{cabin}}$) clamp = 1.0. Daha hassas kullanım için kabin köşelerinin manuel kalibrasyonu (homografi) gerekir → bu repo'da `src/perception/homography.py` + `BEVMaskOccupancy` zaten hazır, ileride aktive edilebilir.

In [ ]:
# --- Class-based area estimation (literature-backed) ---
from dataclasses import dataclass
from typing import Sequence

# Literatürden alinmis ortalama footprint degerleri (m^2)
# Hucre 8.2'deki tabloyla birebir uyumlu. configs/default.yaml ile sync edilebilir.
CLASS_AREAS_M2 = {
    'person':   0.20,   # TS EN 81-20:2020 §5.4.2.1.1 / ISO 8100-1
    'stroller': 0.45,   # ~90x50 cm (single stroller average)
    'luggage':  0.18,   # IATA 56x36 cm + medium check-in mix
    'box':      0.20,   # ~50x40 cm orta boy karton
}

# Asansor zemin alani — configs/default.yaml ile uyumlu (1.4 x 1.6 m = 2.24 m^2).
# Farkli asansor icin override edilebilir.
ELEVATOR_FLOOR_AREA_M2 = 2.24

# Asansor kontrol esikleri (configs/default.yaml -> thresholds)
AREA_BYPASS_RATIO = 0.90   # >%90 doluysa cagri bypass


@dataclass
class AreaReport:
    counts: dict           # {'person': 4, 'stroller': 1, ...}
    occupied_m2: float     # toplam tahmini dolu alan
    cabin_m2: float        # kabin zemin alani
    occupancy_ratio: float # [0, 1]
    breakdown_m2: dict     # {'person': 0.80, 'stroller': 0.45, ...}
    bypass: bool           # area_bypass_ratio asildi mi?

    def pretty(self) -> str:
        lines = [
            f'Doluluk: {self.occupied_m2:.2f} / {self.cabin_m2:.2f} m^2 '
            f'(%{self.occupancy_ratio*100:.1f})'
        ]
        for cls, n in self.counts.items():
            lines.append(f'  • {cls:<9} ×{n}  →  {self.breakdown_m2[cls]:.2f} m^2')
        if self.bypass:
            lines.append(f'  ⚠ AREA BYPASS (>%{AREA_BYPASS_RATIO*100:.0f}) — cagri reddedilmeli')
        return '\n'.join(lines)


def estimate_area(
    detections,                       # list of objects with .class_name attribute
    class_areas_m2: dict = CLASS_AREAS_M2,
    cabin_m2: float = ELEVATOR_FLOOR_AREA_M2,
) -> AreaReport:
    """Class-count x average footprint area estimation.

    Returns AreaReport with per-class counts, total occupied area, and
    occupancy ratio (clamped to 1.0).
    """
    counts: dict[str, int] = {}
    breakdown: dict[str, float] = {}
    occupied = 0.0

    for det in detections:
        cls = det.class_name
        avg = class_areas_m2.get(cls, 0.0)
        if avg <= 0:
            continue
        counts[cls] = counts.get(cls, 0) + 1
        breakdown[cls] = breakdown.get(cls, 0.0) + avg
        occupied += avg

    ratio = min(occupied / cabin_m2, 1.0) if cabin_m2 > 0 else 0.0
    return AreaReport(
        counts=counts,
        occupied_m2=occupied,
        cabin_m2=cabin_m2,
        occupancy_ratio=ratio,
        breakdown_m2=breakdown,
        bypass=(ratio >= AREA_BYPASS_RATIO),
    )


# Sanity test: yapay tespitlerle
from collections import namedtuple
FakeDet = namedtuple('FakeDet', 'class_name')
fake = [FakeDet('person')] * 4 + [FakeDet('stroller')] * 1 + [FakeDet('luggage')] * 2
print(estimate_area(fake).pretty())

## 9. Demo inference + alan raporu (test set'ten 6 örnek)
Her görselde:
- YOLO tahminlerinin görselleştirmesi (sol)
- Sınıf sayıları + alan hesabı raporu (alt başlık)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import glob
from src.detection.detector import Detection

test_imgs = sorted(glob.glob(f'{DATA}/test/images/*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, test_imgs):
    res = model_v2.predict(img_path, conf=0.4, verbose=False)[0]

    # ultralytics result -> Detection listesi
    dets = []
    if res.boxes is not None:
        for box in res.boxes:
            cls_id = int(box.cls.item())
            cls_name = res.names[cls_id]
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            dets.append(Detection(
                class_id=cls_id,
                class_name=cls_name,
                confidence=float(box.conf.item()),
                bbox=(int(x1), int(y1), int(x2), int(y2)),
            ))

    report = estimate_area(dets)

    ax.imshow(Image.fromarray(res.plot()[..., ::-1]))
    title = (
        os.path.basename(img_path)[:32] + '\n' +
        f'Doluluk: %{report.occupancy_ratio*100:.1f}  '
        f'({report.occupied_m2:.2f}/{report.cabin_m2:.2f} m²)'
        + ('  ⚠BYPASS' if report.bypass else '')
    )
    ax.set_title(title, fontsize=9)
    ax.axis('off')

    # Detayli rapor stdout'a
    print(f'\n=== {os.path.basename(img_path)} ===')
    print(report.pretty())

plt.tight_layout()
plt.show()

## 10. Toplu alan raporu — tüm test set
İstatistik: kaç görüntüde doluluk eşiği aşılıyor, ortalama doluluk, vb. Tezdeki değerlendirme bölümü için kullanılabilir.

In [ ]:
import numpy as np

all_test = sorted(glob.glob(f'{DATA}/test/images/*.jpg'))
ratios = []
bypass_count = 0
class_total = {k: 0 for k in CLASS_AREAS_M2}

for img_path in all_test:
    res = model_v2.predict(img_path, conf=0.4, verbose=False)[0]
    dets = []
    if res.boxes is not None:
        for box in res.boxes:
            cls_id = int(box.cls.item())
            cls_name = res.names[cls_id]
            dets.append(Detection(
                class_id=cls_id, class_name=cls_name,
                confidence=float(box.conf.item()),
                bbox=(0, 0, 1, 1),  # bbox burada onemsiz
            ))
    rep = estimate_area(dets)
    ratios.append(rep.occupancy_ratio)
    if rep.bypass:
        bypass_count += 1
    for k, v in rep.counts.items():
        class_total[k] = class_total.get(k, 0) + v

ratios = np.array(ratios)
print(f'Test goruntu sayisi:   {len(ratios)}')
print(f'Ortalama doluluk:      %{ratios.mean()*100:.1f}')
print(f'Median doluluk:        %{np.median(ratios)*100:.1f}')
print(f'Max doluluk:           %{ratios.max()*100:.1f}')
print(f'Bypass tetikleyen:     {bypass_count} / {len(ratios)} '
      f'(%{bypass_count/len(ratios)*100:.1f})')
print(f'\nToplam tespit dagilimi:')
for k, v in class_total.items():
    print(f'  • {k:<9} ×{v}')

# Histogram
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ratios * 100, bins=20, edgecolor='black', alpha=0.75)
ax.axvline(AREA_BYPASS_RATIO * 100, color='red', linestyle='--',
           label=f'Bypass esigi (%{AREA_BYPASS_RATIO*100:.0f})')
ax.set_xlabel('Doluluk (%)')
ax.set_ylabel('Goruntu sayisi')
ax.set_title('Test set uzerinde tahmini asansor doluluk dagilimi')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Sonraki adımlar / İleride geliştirilebilir

Tezdeki **"Future Work" bölümü için kullanılabilir notlar**:

1. **Homografi kalibrasyonu** — Asansör kabininin 4 köşesi etiketlenip `cv2.findHomography` ile zemin düzlemine projeksiyon yapılabilir. Bu durumda `src/perception/occupancy.py:FootprintOccupancy` veya `BEVMaskOccupancy` kullanıma alınır → kişi başı sabit 0.20 m² yerine **gerçek konuma göre union of disks** hesabı.
2. **Pose estimation** — Asansör tavanına ek olarak ön-yan kamera eklenirse YOLOv8-pose ile full-body keypoint çıkarılabilir, bbox alanı yerine vücut silüetinin gerçek piksel maskesi kullanılır.
3. **Multi-frame tracking** — Bir kişi 2-3 ardışık frame'de farklı conf seviyelerinde tespit ediliyorsa BoT-SORT/ByteTrack ile track ID'si verilip alan hesabına 1 kez katkı sağlanabilir (overcounting önleme).
4. **Per-class footprint dağılımı** — Sabit ortalama yerine sınıf içi varyans modellenebilir (ör. "large stroller" 0.65 m², "compact stroller" 0.30 m² ayrı sınıf).
5. **Etiket genişletme** — Yeni dataset toplanırken kameraya yakın kişiler için "head+shoulder" etiketi yerine "head+shoulder+visible-torso" şeklinde tutarlı genişletilmiş etiket politikası → bbox piksel alanı tabanlı yöntem kameradan bağımsız olarak çalışabilir.

---

## 12. Yerele indirme
1. **`best_v2.pt`** → Drive senkronizasyonu ile yerel makineye iner: `Desktop/Capstone_deneme_ai/models/weights/best.pt`.
2. Notebook 03 (BEV demo) ve Notebook 04 (enerji simülasyonu) yeni `best.pt`'yi kullanmaya devam eder.
3. v1'e geri dönmek istersen: `MyDrive/Capstone/models/backups/best_v1_*_backup_2026-05-07.pt` dosyasını `best.pt` olarak kopyala.